# Эксперимент 03 — сравнение моделей
**Цель:** сравнить baseline и улучшенные модели.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np
from src.config import load_config
from src.data import load_csv, prepare_training_data
from src.features import MODEL_FEATURES
from src.train import build_models, build_preprocessor, evaluate
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [ ]:
config = load_config()
raw = load_csv(config['paths']['train_data'], max_rows=300000)
data = prepare_training_data(raw, config['filters'])
X = data[MODEL_FEATURES]
y_log = np.log1p(data['trip_duration'])
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

In [ ]:
results = []
for name, estimator in build_models(42).items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', estimator)])
    pipe.fit(X_train, y_train)
    results.append({'model': name, **evaluate(pipe, X_test, y_test)})
results_df = pd.DataFrame(results).sort_values('rmsle')
results_df